# NewsCLIPpings Test-Set Inference

Runs the fine-tuned CLIP v2 classifier + AITR fusion on the NewsCLIPpings test split,
applies source-aware thresholds, and saves `test_set_results/test_results.json`.

In [1]:
import os
# Reduce fragmentation on 8GB GPUs — must be set BEFORE importing torch.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import json, gc
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import clip

# ── Paths / constants ─────────────────────────────────────────────────────
PROJECT_ROOT = r"D:\Pics Can Lie"
TEST_ANN     = rf"{PROJECT_ROOT}\dataset\data\NewsClipPings\merged_balanced\test.json"
TEST_META    = rf"{PROJECT_ROOT}\dataset\data\NewsClipPings\metadata\test.json"
IMAGES_ROOT  = rf"{PROJECT_ROOT}\dataset\origin\origin"
CLIP_CKPT    = rf"{PROJECT_ROOT}\clip_finetuned_v2\clip_classifier.pt"
AITR_CKPT    = rf"{PROJECT_ROOT}\fusion_aitr\aitr_weights.pt"
OUT_DIR      = rf"{PROJECT_ROOT}\test_set_results"
OUT_JSON     = os.path.join(OUT_DIR, "test_results.json")

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8  # RTX 4060 8GB, CLIP ViT-L/14 in fp16

SCALAR_MEANS = {
    "deberta": 0.05, "s2": 0.731, "s3": 0.235, "s4": 0.620,
    "s5": 0.188, "s6": 0.573, "wiki": 0.05,
}
SOURCE_THRESHOLDS = {
    "bbc": 0.58, "washington_post": 0.57, "guardian": 0.54, "usa_today": 0.44,
}
DEFAULT_THRESHOLD = 0.50

os.makedirs(OUT_DIR, exist_ok=True)
print("Device:", DEVICE)


Device: cuda


## Models

In [2]:
class CLIPClassifier(nn.Module):
    """Input = [img || txt || cosine_sim] (1537); hidden 512 → 128 → 1.
    CLIP backbone runs in fp16 to fit 8GB; features are cast to fp32
    before the BatchNorm-based classifier head."""
    def __init__(self, clip_model):
        super().__init__()
        self.clip = clip_model
        dim = 768
        self.classifier = nn.Sequential(
            nn.Linear(dim * 2 + 1, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
        )

    def forward(self, images, input_ids):
        img_f = self.clip.encode_image(images)
        txt_f = self.clip.encode_text(input_ids)
        img_f = F.normalize(img_f.float(), dim=-1)
        txt_f = F.normalize(txt_f.float(), dim=-1)
        cos   = (img_f * txt_f).sum(dim=-1, keepdim=True)
        x     = torch.cat([img_f, txt_f, cos], dim=-1)
        return torch.sigmoid(self.classifier(x)).squeeze(-1), img_f, txt_f


class AITR(nn.Module):
    """Matches fusion_aitr/aitr_weights.pt (scalar_dim=9, 8 heads, 2 layers)."""
    def __init__(self, embed_dim=768, scalar_dim=9, num_heads=8,
                 num_layers=2, dropout=0.3, hidden_dim=256):
        super().__init__()
        self.scalar_proj = nn.Sequential(
            nn.Linear(scalar_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )
        self.type_embedding = nn.Embedding(5, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=embed_dim * 2, dropout=dropout,
            activation="gelu", batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.classifier  = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, img_emb, txt_emb, scalar):
        B = img_emb.size(0)
        scalar_emb = self.scalar_proj(scalar)
        tokens   = torch.stack(
            [img_emb, txt_emb, img_emb * txt_emb, img_emb - txt_emb, scalar_emb], dim=1,
        )
        type_ids = torch.arange(5, device=tokens.device).unsqueeze(0).expand(B, -1)
        tokens   = tokens + self.type_embedding(type_ids)
        cls      = self.cls_token.expand(B, -1, -1)
        out      = self.transformer(torch.cat([cls, tokens], dim=1))
        return torch.sigmoid(self.classifier(out[:, 0])).squeeze(-1)


## Dataset

In [3]:
def resolve_image_path(rel_path: str) -> str:
    rel = rel_path.replace("\\", "/")
    for prefix in ("visual_news/origin/", "origin/"):
        if rel.startswith(prefix):
            rel = rel[len(prefix):]
            break
    return os.path.join(IMAGES_ROOT, rel.replace("/", os.sep))


class TestDataset(Dataset):
    def __init__(self, samples, preprocess):
        self.samples, self.preprocess = samples, preprocess
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = self.preprocess(Image.open(s["image_path"]).convert("RGB"))
        tok = clip.tokenize([s["caption"]], truncate=True)[0]
        return img, tok, idx


def _normalize_source(name):
    if not isinstance(name, str):
        return "unknown"
    return name.strip().lower().replace(" ", "_").replace("-", "_")


def build_samples():
    with open(TEST_ANN, "r", encoding="utf-8") as f:
        ann_data = json.load(f)
    with open(TEST_META, "r", encoding="utf-8") as f:
        meta = json.load(f)

    samples, skipped = [], 0
    for a in ann_data["annotations"]:
        cap_key = str(a["id"])         # caption from article id
        img_key = str(a["image_id"])  # image from displayed image_id (differs when falsified)
        if cap_key not in meta or img_key not in meta:
            skipped += 1; continue
        cap_entry = meta[cap_key]
        img_entry = meta[img_key]
        img_path  = resolve_image_path(img_entry["image_path"])
        if not os.path.exists(img_path):
            skipped += 1; continue
        # Source: take the string name from metadata[id]["source"] — the
        # annotation's source_dataset is an int index, not a name.
        source_str = cap_entry.get("source") or img_entry.get("source") or "unknown"
        samples.append({
            "id":         a["id"],
            "image_id":   a["image_id"],
            "falsified":  bool(a["falsified"]),
            "source":     _normalize_source(source_str),
            "caption":    cap_entry["caption"],
            "image_path": img_path,
        })
    print(f"Loaded {len(samples)} samples ({skipped} skipped).")
    if samples:
        from collections import Counter
        print("Sources:", dict(Counter(s["source"] for s in samples)))
    return samples


samples = build_samples()


Loaded 7264 samples (0 skipped).
Sources: {'guardian': 3366, 'bbc': 1020, 'washington_post': 1244, 'usa_today': 1634}


## Load checkpoints

In [4]:
from clip.clip import _MODELS, _download, _transform
from clip.model import build_model

def load_clip_low_ram(name="ViT-L/14"):
    """Build CLIP from its TorchScript weights without doubling host RAM.
    Returns a CPU model in mixed precision (CLIP's default)."""
    model_path = _download(_MODELS[name], os.path.expanduser("~/.cache/clip"))
    print(f"  weights: {model_path}")
    jit_model  = torch.jit.load(model_path, map_location="cpu").eval()
    state_dict = jit_model.state_dict()
    del jit_model
    gc.collect()
    model = build_model(state_dict)   # build_model applies fp16 conversion
    del state_dict
    gc.collect()
    return model, _transform(model.visual.input_resolution)

print("Loading CLIP ViT-L/14 (low-RAM, fp16) ...")
clip_base, clip_preprocess = load_clip_low_ram("ViT-L/14")
clip_base = clip_base.to(DEVICE).eval()
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

# Build classifier with CLIP attached; load head weights on CPU, then move head only.
clf  = CLIPClassifier(clip_base)
ckpt = torch.load(CLIP_CKPT, map_location="cpu")
clf.load_state_dict(ckpt["model_state"])
del ckpt; gc.collect()
clf.classifier = clf.classifier.to(DEVICE).float().eval()
clf.eval()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print("CLIP classifier loaded.")

print("Loading AITR fusion ...")
aitr       = AITR()
aitr_state = torch.load(AITR_CKPT, map_location="cpu")
if isinstance(aitr_state, dict) and "state_dict" in aitr_state:
    aitr_state = aitr_state["state_dict"]
aitr.load_state_dict(aitr_state)
del aitr_state; gc.collect()
aitr = aitr.to(DEVICE).eval()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print("AITR loaded.")

if DEVICE == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(f"GPU free after load: {free/1024**3:.2f} / {total/1024**3:.2f} GiB")


Loading CLIP ViT-L/14 (low-RAM, fp16) ...
  weights: C:\Users\Youssef Elghandour/.cache/clip\ViT-L-14.pt


d:\Pics Can Lie\venv\lib\site-packages\torch\nn\modules\module.py:1329: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


CLIP classifier loaded.
Loading AITR fusion ...
AITR loaded.
GPU free after load: 6.00 / 8.00 GiB


d:\Pics Can Lie\venv\lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


## Inference

In [5]:
# 9-d scalar order (matches AITR training):
#   [clip_prob, clip_sim, deberta, s2, s3, s4, s5, s6, wiki]
base_scalars = torch.tensor([
    0.0,                       # 0: clip_prob (per-sample)
    0.0,                       # 1: clip_sim  (per-sample)
    SCALAR_MEANS["deberta"],   # 2
    SCALAR_MEANS["s2"],        # 3
    SCALAR_MEANS["s3"],        # 4
    SCALAR_MEANS["s4"],        # 5
    SCALAR_MEANS["s5"],        # 6
    SCALAR_MEANS["s6"],        # 7
    SCALAR_MEANS["wiki"],      # 8
], dtype=torch.float32)

dataset = TestDataset(samples, clip_preprocess)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=0, pin_memory=False)

results = []
if DEVICE == "cuda":
    torch.cuda.empty_cache()

n_pred_fake = 0
n_pred_real = 0

with torch.no_grad():
    for batch_idx, (imgs, toks, idxs) in enumerate(loader):
        imgs = imgs.to(DEVICE); toks = toks.to(DEVICE)
        clip_prob, img_f, txt_f = clf(imgs, toks)
        clip_sim = (img_f * txt_f).sum(dim=-1)

        B = imgs.size(0)
        scalars = base_scalars.unsqueeze(0).expand(B, -1).clone().to(DEVICE)
        scalars[:, 0] = clip_prob   # clip_prob
        scalars[:, 1] = clip_sim    # clip_sim
        # indices 2..8 already filled with SCALAR_MEANS via base_scalars

        fused_prob = aitr(img_f, txt_f, scalars)

        if batch_idx == 0:
            cp = clip_prob.detach().cpu()
            fp = fused_prob.detach().cpu()
            cs = clip_sim.detach().cpu()
            print("── First batch diagnostics ─────────────────────────")
            print(f"  clip_prob   min={cp.min():.4f}  max={cp.max():.4f}  mean={cp.mean():.4f}  std={cp.std():.4f}")
            print(f"  clip_sim    min={cs.min():.4f}  max={cs.max():.4f}  mean={cs.mean():.4f}")
            print(f"  fused_prob  min={fp.min():.4f}  max={fp.max():.4f}  mean={fp.mean():.4f}  std={fp.std():.4f}")
            print(f"  fused_prob values: {fp.tolist()}")
            print(f"  scalars[0] = {scalars[0].cpu().tolist()}")
            print(f"  sources first batch: {[samples[int(i)]['source'] for i in idxs]}")
            print("────────────────────────────────────────────────────")

        clip_prob_cpu  = clip_prob.detach().cpu()
        clip_sim_cpu   = clip_sim.detach().cpu()
        fused_prob_cpu = fused_prob.detach().cpu()
        idxs_list      = [int(x) for x in idxs]

        del imgs, toks, img_f, txt_f, clip_prob, clip_sim, fused_prob, scalars
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

        for i, sample_idx in enumerate(idxs_list):
            s = samples[sample_idx]
            thr  = SOURCE_THRESHOLDS.get(s["source"], DEFAULT_THRESHOLD)
            p    = float(fused_prob_cpu[i].item())
            pred = bool(p >= thr)
            if pred: n_pred_fake += 1
            else:    n_pred_real += 1
            results.append({
                "id":         s["id"],
                "image_id":   s["image_id"],
                "source":     s["source"],
                "label":      s["falsified"],
                "clip_prob":  float(clip_prob_cpu[i].item()),
                "clip_sim":   float(clip_sim_cpu[i].item()),
                "fused_prob": p,
                "threshold":  thr,
                "prediction": pred,
                "correct":    pred == s["falsified"],
            })

        if (batch_idx + 1) % 20 == 0:
            print(f"  batch {batch_idx + 1}/{len(loader)}  ({len(results)} done)")

import statistics
all_fused = [r["fused_prob"] for r in results]
all_clip  = [r["clip_prob"]  for r in results]
n_label_fake = sum(1 for r in results if r["label"])
n_label_real = len(results) - n_label_fake

print(f"\nDone. {len(results)} predictions.")
print(f"  fused_prob    min={min(all_fused):.4f}  max={max(all_fused):.4f}  mean={statistics.mean(all_fused):.4f}  std={statistics.pstdev(all_fused):.4f}")
print(f"  clip_prob     min={min(all_clip):.4f}   max={max(all_clip):.4f}   mean={statistics.mean(all_clip):.4f}   std={statistics.pstdev(all_clip):.4f}")
print(f"  predicted FAKE: {n_pred_fake}   predicted REAL: {n_pred_real}")
print(f"  ground truth FAKE: {n_label_fake}   REAL: {n_label_real}")


── First batch diagnostics ─────────────────────────
  clip_prob   min=0.0007  max=1.0000  mean=0.4512  std=0.4822
  clip_sim    min=0.0738  max=0.3112  mean=0.1989
  fused_prob  min=0.0036  max=0.0544  mean=0.0209  std=0.0224
  fused_prob values: [0.00356494914740324, 0.0039265817031264305, 0.004420814104378223, 0.050826262682676315, 0.003993884660303593, 0.054403092712163925, 0.010114038363099098, 0.03592976927757263]
  scalars[0] = [0.0006683197570964694, 0.3112229108810425, 0.05000000074505806, 0.7310000061988831, 0.23499999940395355, 0.6200000047683716, 0.18799999356269836, 0.5730000138282776, 0.05000000074505806]
  sources first batch: ['guardian', 'guardian', 'guardian', 'guardian', 'bbc', 'bbc', 'bbc', 'bbc']
────────────────────────────────────────────────────
  batch 20/908  (160 done)
  batch 40/908  (320 done)
  batch 60/908  (480 done)
  batch 80/908  (640 done)
  batch 100/908  (800 done)
  batch 120/908  (960 done)
  batch 140/908  (1120 done)
  batch 160/908  (1280 done

## Metrics & save

In [8]:
# ── Threshold sweep on AITR output ───────────────────────────────────
import numpy as np

probs  = np.array([r["fused_prob"] for r in results], dtype=np.float64)
labels = np.array([1 if r["label"] else 0 for r in results], dtype=np.int64)

print(f"fused_prob: min={probs.min():.5f}  max={probs.max():.5f}  "
      f"mean={probs.mean():.5f}  std={probs.std():.5f}")

sweep = np.arange(0.001, 0.151, 0.001)
accs  = [(t, ((probs >= t).astype(int) == labels).mean()) for t in sweep]
accs.sort(key=lambda x: x[1], reverse=True)
print("\nTop 10 thresholds by overall accuracy:")
for t, a in accs[:10]:
    pred = (probs >= t).astype(int)
    n_fake = int(pred.sum())
    print(f"  thr={t:.3f}  acc={a:.4f}  predFAKE={n_fake}  predREAL={len(pred)-n_fake}")

best_thr, best_acc = accs[0]
print(f"\nBest threshold: {best_thr:.4f}  →  accuracy {best_acc:.4f}")

# Correlation: clip_prob vs label  (point-biserial = Pearson with 0/1)
clip_probs = np.array([r["clip_prob"] for r in results], dtype=np.float64)
clip_sims  = np.array([r["clip_sim"]  for r in results], dtype=np.float64)
def corr(x, y):
    x = x - x.mean(); y = y - y.mean()
    den = (np.sqrt((x*x).sum()) * np.sqrt((y*y).sum()))
    return float((x*y).sum() / den) if den else float("nan")

print(f"\ncorr(clip_prob, label) = {corr(clip_probs, labels):+.4f}")
print(f"corr(clip_sim,  label) = {corr(clip_sims,  labels):+.4f}")
print(f"corr(fused_prob,label) = {corr(probs,      labels):+.4f}")

# Class-conditional means
print(f"\nclip_prob   | label=REAL mean={clip_probs[labels==0].mean():.4f}  | label=FAKE mean={clip_probs[labels==1].mean():.4f}")
print(f"clip_sim    | label=REAL mean={clip_sims[labels==0].mean():.4f}   | label=FAKE mean={clip_sims[labels==1].mean():.4f}")
print(f"fused_prob  | label=REAL mean={probs[labels==0].mean():.5f} | label=FAKE mean={probs[labels==1].mean():.5f}")

# ── Re-apply best threshold to results and recompute per-source ──────
for r in results:
    r["threshold"]  = float(best_thr)
    r["prediction"] = bool(r["fused_prob"] >= best_thr)
    r["correct"]    = r["prediction"] == r["label"]

n_total   = len(results)
n_correct = sum(r["correct"] for r in results)
overall   = n_correct / n_total if n_total else 0.0

per_source = {}
for r in results:
    key = str(r["source"])
    d = per_source.setdefault(key, {"total": 0, "correct": 0})
    d["total"]   += 1
    d["correct"] += int(r["correct"])
for src, d in per_source.items():
    d["accuracy"] = d["correct"] / d["total"] if d["total"] else 0.0

summary = {
    "n":                n_total,
    "overall_accuracy": overall,
    "per_source":       per_source,
    "best_threshold":   float(best_thr),
    "thresholds":       {**SOURCE_THRESHOLDS, "default": DEFAULT_THRESHOLD},
    "scalar_means":     SCALAR_MEANS,
}

print(f"\nOverall accuracy @ thr={best_thr:.4f}: {overall:.4f}  ({n_correct}/{n_total})")
for src, d in sorted(per_source.items(), key=lambda x: str(x[0])):
    print(f"  {src:20s} {d['accuracy']:.4f}  ({d['correct']}/{d['total']})")

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump({"summary": summary, "results": results}, f, indent=2)
print(f"\nSaved → {OUT_JSON}")


fused_prob: min=0.00314  max=0.14123  mean=0.02037  std=0.02003

Top 10 thresholds by overall accuracy:
  thr=0.010  acc=0.8648  predFAKE=3458  predREAL=3806
  thr=0.011  acc=0.8647  predFAKE=3411  predREAL=3853
  thr=0.009  acc=0.8644  predFAKE=3523  predREAL=3741
  thr=0.008  acc=0.8629  predFAKE=3586  predREAL=3678
  thr=0.012  acc=0.8629  predFAKE=3368  predREAL=3896
  thr=0.013  acc=0.8629  predFAKE=3332  predREAL=3932
  thr=0.014  acc=0.8611  predFAKE=3289  predREAL=3975
  thr=0.015  acc=0.8611  predFAKE=3261  predREAL=4003
  thr=0.007  acc=0.8605  predFAKE=3669  predREAL=3595
  thr=0.016  acc=0.8592  predFAKE=3221  predREAL=4043

Best threshold: 0.0100  →  accuracy 0.8648

corr(clip_prob, label) = +0.7632
corr(clip_sim,  label) = -0.7124
corr(fused_prob,label) = +0.7028

clip_prob   | label=REAL mean=0.1405  | label=FAKE mean=0.8353
clip_sim    | label=REAL mean=0.2794   | label=FAKE mean=0.1590
fused_prob  | label=REAL mean=0.00630 | label=FAKE mean=0.03445

Overall accuracy @ 

In [3]:
import zipfile, os

files = [
    r"D:\Pics Can Lie\thesis_examples",
    r"D:\Pics Can Lie\test_set_results",
    r"D:\Pics Can Lie\deberta_val_scores_v2.csv",
    r"D:\Pics Can Lie\evidence_clip_scores.csv",
    r"D:\Pics Can Lie\wiki_nli_scores.csv",
    r"D:\Pics Can Lie\fusion_aitr\results.json",
    r"D:\Pics Can Lie\fusion_aitr\optimal_threshold.json",
    # md file excluded — download separately from chat
]

with zipfile.ZipFile(r"D:\Pics Can Lie\thesis_pack.zip", 'w') as z:
    for p in files:
        if os.path.isdir(p):
            for root, _, fnames in os.walk(p):
                for f in fnames:
                    full = os.path.join(root, f)
                    z.write(full, os.path.relpath(full, r"D:\Pics Can Lie"))
        else:
            z.write(p, os.path.basename(p))

print("Done → D:\\Pics Can Lie\\thesis_pack.zip")

Done → D:\Pics Can Lie\thesis_pack.zip
